# Test Set Synthetic Generation

Generate 10 synthetic call center conversations per language pair using OpenAI API.
- Positive: complete conversation with `<|im_end|>`
- Negative: truncated at 0.2-0.7 of last message, no `<|im_end|>`

In [ ]:
import os
import json
import random
from openai import OpenAI
from tqdm import tqdm

os.environ['OPENAI_API_KEY'] = 'sk-proj-xxx'  # <-- paste your key here
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

LANGUAGES = [
    'chinese-english',
    'chinese-malay',
    'chinese-tamil',
    'english-chinese',
    'english-malay',
    'english-tamil',
    'malay-chinese',
    'malay-english',
    'malay-tamil',
    'tamil-chinese',
    'tamil-english',
    'tamil-malay',
]

CONVOS_PER_LANG = 10
MODEL = 'gpt-4o-mini'

print(f'Languages: {len(LANGUAGES)}')
print(f'Conversations per language: {CONVOS_PER_LANG}')
print(f'Total conversations: {len(LANGUAGES) * CONVOS_PER_LANG}')

In [ ]:
LANG_NAMES = {
    'chinese': 'Mandarin Chinese',
    'english': 'English',
    'malay': 'Malay (Bahasa Melayu)',
    'tamil': 'Tamil',
}

TOPICS = [
    'billing inquiry',
    'account update',
    'service complaint',
    'technical support',
    'insurance claim',
    'loan application',
    'appointment scheduling',
    'product return',
    'subscription cancellation',
    'payment issue',
    'address change',
    'password reset',
    'delivery tracking',
    'refund request',
    'new account opening',
]


def generate_conversation(lang_pair, topic):
    lang1, lang2 = lang_pair.split('-')
    lang1_name = LANG_NAMES[lang1]
    lang2_name = LANG_NAMES[lang2]

    prompt = f"""Generate a realistic call center conversation between a customer (caller) and an agent.

Rules:
- The customer speaks primarily in {lang1_name}, occasionally mixing in {lang2_name} words/phrases (code-switching)
- The agent responds primarily in {lang1_name}, also mixing in {lang2_name} where natural
- Topic: {topic}
- ONLY 1-2 turns total (1 turn = customer message + agent response). Keep it short.
- Make it sound natural, like a real Malaysian/Southeast Asian call center
- The conversation should reach a natural conclusion

Return ONLY a JSON array of messages, each with "role" (either "assistance" or "assistant") and "content" fields.
- First message role should be "assistance" (the customer)
- Alternate between "assistance" (customer) and "assistant" (agent)
- Do NOT include system messages

Example format:
[{{"role": "assistance", "content": "customer message..."}}, {{"role": "assistant", "content": "agent response..."}}]"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.9,
        max_tokens=2000,
    )

    text = response.choices[0].message.content.strip()
    # Strip markdown code block if present
    if text.startswith('```'):
        text = text.split('\n', 1)[1]
        text = text.rsplit('```', 1)[0]
    
    messages = json.loads(text)
    return messages


print('Ready')

In [ ]:
conversations = []

for lang_pair in tqdm(LANGUAGES, desc='Languages'):
    topics = random.sample(TOPICS, CONVOS_PER_LANG)
    for i, topic in enumerate(topics):
        try:
            messages = generate_conversation(lang_pair, topic)
            conversations.append({
                'language': lang_pair,
                'topic': topic,
                'messages': messages,
            })
            print(f'  {lang_pair} [{i+1}/{CONVOS_PER_LANG}] {topic} -> {len(messages)} turns')
        except Exception as e:
            print(f'  {lang_pair} [{i+1}/{CONVOS_PER_LANG}] FAILED: {e}')

print(f'\nTotal conversations: {len(conversations)}')

In [ ]:
# Preview some conversations
for conv in conversations[:3]:
    print(f"\n=== {conv['language']} | {conv['topic']} ===")
    for msg in conv['messages']:
        print(f"  [{msg['role']}] {msg['content'][:100]}...")

## Build positive and negative samples

In [ ]:
def format_chat_template(messages):
    parts = []
    for msg in messages:
        parts.append(f'<|im_start|>{msg["role"]}\n{msg["content"]}<|im_end|>')
    return '\n'.join(parts)


def build_positive(messages):
    """Complete conversation ending with <|im_end|>."""
    return format_chat_template(messages)


def build_negative(messages):
    """Truncated conversation, no <|im_end|> at end."""
    text = format_chat_template(messages)

    # Strip trailing <|im_end|>
    if text.endswith('<|im_end|>'):
        text = text[:-len('<|im_end|>')]

    # Find last <|im_start|> block and truncate within it
    last_im_start = text.rfind('<|im_start|>')
    if last_im_start >= 0:
        newline_after_role = text.find('\n', last_im_start)
        if newline_after_role >= 0:
            content_start = newline_after_role + 1
            content = text[content_start:]
            if len(content) > 10:
                keep_ratio = random.uniform(0.2, 0.7)
                keep_len = max(5, int(len(content) * keep_ratio))
                text = text[:content_start + keep_len]

    return text.rstrip()


# Build test samples
test_samples = []
for conv in conversations:
    messages = conv['messages']
    if not messages or len(messages) < 2:
        continue
    test_samples.append({
        'text': build_positive(messages),
        'label': 'positive',
        'language': conv['language'],
    })
    test_samples.append({
        'text': build_negative(messages),
        'label': 'negative',
        'language': conv['language'],
    })

print(f'Total test samples: {len(test_samples)}')
print(f'  Positive: {sum(1 for s in test_samples if s["label"] == "positive")}')
print(f'  Negative: {sum(1 for s in test_samples if s["label"] == "negative")}')

In [ ]:
# Verify samples look correct
for s in test_samples[:6]:
    text = s['text']
    ends_with_imend = text.endswith('<|im_end|>')
    print(f"[{s['label']:>8}] lang={s['language']:<20} ends_with_imend={ends_with_imend} len={len(text)}")
    print(f"  last 80 chars: ...{text[-80:]}")
    print()

## Save

In [ ]:
# Save raw conversations
with open('synthetic_conversations.json', 'w') as f:
    json.dump(conversations, f, ensure_ascii=False, indent=2)
print(f'Saved {len(conversations)} conversations to synthetic_conversations.json')

# Save test samples
with open('synthetic_test_samples.json', 'w') as f:
    json.dump(test_samples, f, ensure_ascii=False, indent=2)
print(f'Saved {len(test_samples)} test samples to synthetic_test_samples.json')